In [ ]:
import json
import requests
import pandas as pd
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

In [ ]:
# Environment variables - loaded from .env file
tableauServerName = os.getenv('TABLEAU_SERVER_NAME')
siteId = os.getenv('TABLEAU_SITE_ID')

# Datasource LUIDs from environment variables
california_schools_fprm_satscores = os.getenv('TABLEAU_DS_FPRM_SATSCORES')
california_schools_frpm_schools = os.getenv('TABLEAU_DS_FRPM_SCHOOLS')
california_schools_satscores_schools = os.getenv('TABLEAU_DS_SATSCORES_SCHOOLS')

# Personal Access Token (PAT) from environment variables
patName = os.getenv('TABLEAU_PAT_NAME')
patSecret = os.getenv('TABLEAU_PAT_SECRET')

# VizQL Data Service endpoints
vds_path = '/api/v1/vizql-data-service/query-datasource'
signin_path = '/api/3.24/auth/signin'



In [ ]:
# Authenticate with Tableau to get auth token
def get_auth_token():
    """Sign in to Tableau using PAT and get authentication token"""
    url = f"{tableauServerName}{signin_path}"
    
    payload = json.dumps({
        "credentials": {
            "personalAccessTokenName": patName,
            "personalAccessTokenSecret": patSecret,
            "site": {
                "contentUrl": siteId
            }
        }
    })
    
    headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    }
    
    response = requests.post(url, headers=headers, data=payload)
    
    if response.status_code == 200:
        token = response.json()['credentials']['token']
        print(f"Successfully authenticated!")
        return token
    else:
        print(f"Authentication failed. Status code: {response.status_code}")
        print(response.text)
        return None

# Get the authentication token
auth_token = get_auth_token()
auth_token

In [ ]:
#define the headless BI query template
def send(datasourceLuid, query):
    """Query VizQL Data Service with the authenticated token"""
    # Build the full URL according to VizQL Data Service spec
    # Format: https://{your-pod}.online.tableau.com/api/v1/vizql-data-service/query-datasource
    url = f"{tableauServerName}{vds_path}"
    
    # Build the payload according to VizQL Data Service API spec
    payload = json.dumps({
        "datasource": {
            "datasourceLuid": datasourceLuid
        },
        "query": query
    })

    # Build the headers with proper authentication using the token from signin
    headers = {
        'X-Tableau-Auth': auth_token,
        'Content-Type': 'application/json'
    }
    
    response = requests.post(url, headers=headers, data=payload)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        data = response.json()['data']
        # Create a pandas DataFrame from the JSON data
        df = pd.DataFrame(data)
        return df
    else:
        print("Failed to fetch data from the API. Status code:", response.status_code)
        print(response.text)
        return None


# VDS Query Test Cases from mini_dev_vds.json

This section tests each VDS query defined in the benchmark file.


In [ ]:
# Load the VDS queries from JSON file
with open('mini_dev_postgresql_vds.json', 'r') as f:
    vds_test_cases = json.load(f)

# Create a lookup dictionary by question_id
vds_queries = {tc['question_id']: tc for tc in vds_test_cases}
print(f"Loaded {len(vds_queries)} test cases")

## Question 5: Virtual schools with Math > 400


In [ ]:
test_case = vds_queries[5]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 11: School codes with enrollment > 500


In [ ]:
test_case = vds_queries[11]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 12: Highest eligible free rate (excellence rate > 0.3)


In [ ]:
test_case = vds_queries[12]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_fprm_satscores, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 17: Rank schools by writing score > 499


In [ ]:
test_case = vds_queries[17]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 23: Schools with enrollment difference > 30


In [ ]:
test_case = vds_queries[23]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 24: Schools with free meal rate > 0.1 and test scores >= 1500


In [ ]:
test_case = vds_queries[24]

print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_fprm_satscores, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 25: Riverside schools avg math > 400 


In [ ]:
test_case = vds_queries[25]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_fprm_satscores, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 26: Monterey high schools with >800 free meals


In [ ]:
test_case = vds_queries[26]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools,test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 27: Schools opened after 1991 or closed before 2000 


In [ ]:
test_case = vds_queries[27]
#print(test_case)

print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 28: Locally funded schools above avg enrollment


In [ ]:
test_case = vds_queries[28]

print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 31: Free rate of 10th & 11th highest enrollment


In [ ]:
test_case = vds_queries[31]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_fprm_satscores, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 32: Top 5 FRPM rates for SOC=66 schools


In [ ]:
test_case = vds_queries[32]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 37: Address of school with lowest excellence rate (NOT SUPPORTED)


In [ ]:
test_case = vds_queries[37]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 36: School admins with highest NumGE1500 (Not Supported)


In [ ]:
# test_case = vds_queries[36]
# print(f"Question: {test_case['question']}")
# print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

# if 'VDS_QUERY' in test_case:
#     df = send(test_case['VDS_QUERY'])
#     display(df)
# else:
#     print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 39: Avg test takers from Fresno schools opened in 1980


In [ ]:
test_case = vds_queries[39]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 40: Phone of Fresno Unified school with lowest reading score


In [ ]:
test_case = vds_queries[40]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 41: Virtual schools top 5 by county (NOT SUPPORTED)


In [ ]:
test_case = vds_queries[41]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 45: Schools managed by Ricci Ulrich with writing scores


In [ ]:
test_case = vds_queries[45]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 46: State special school with highest K-12 enrollment


In [ ]:
test_case = vds_queries[46]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools,test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 47: Monthly avg schools opened in Alameda 1980 (NOT SUPPORTED)


In [ ]:
test_case = vds_queries[47]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 48: Ratio merged DOC 54 to DOC 52 in Orange County


In [ ]:
test_case = vds_queries[48]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 50: 7th highest math score school address


In [ ]:
test_case = vds_queries[50]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 62: Non-charter LA schools with free meal % < 0.18


In [ ]:
test_case = vds_queries[62]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 72: SSS school enrollment in Fremont 2014-2015


In [ ]:
test_case = vds_queries[72]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 77: LA schools with K-9 grade span + FRPM %


In [ ]:
test_case = vds_queries[77]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 79: San Diego vs Santa Barbara virtual school count


In [ ]:
test_case = vds_queries[79]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_satscores_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 82: Grade span of school with highest longitude


In [ ]:
test_case = vds_queries[82]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 83: Magnet K-8 schools with Multiple Provision Types (NOT SUPPORTED)


In [ ]:
test_case = vds_queries[83]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 85: Free meal % for Alusine-administered school


In [ ]:
test_case = vds_queries[85]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")


## Question 87: San Bernardino admin emails


In [ ]:
test_case = vds_queries[87]
print(f"Question: {test_case['question']}")
print(f"Evidence: {test_case.get('evidence', 'N/A')}\n")

if 'VDS_QUERY' in test_case:
    df = send(california_schools_frpm_schools, test_case['VDS_QUERY'])
    display(df)
else:
    print(f"VDS_NOTE: {test_case.get('VDS_NOTE', 'No VDS query available')}")
